# Grover's Algorithm — Hands-On Workshop (HO#02)

This notebook takes you from the classical search problem, through how a quantum computer can query an oracle, to building and running Grover's algorithm yourself — and finally guessing the contents of two **secret** oracles using nothing but what you observe.

**Sections:**
1. Intro to the oracle search problem
2. Classical oracle — worst case (shown)
3. Quantum oracle — demo (shown)
4. Anatomy of Grover's algorithm
5. Build Grover layer-by-layer (hands-on, core)
6. Guess the secret oracles (hands-on, core, graded)
7. *(Optional, self-paced)* Build your own oracle from scratch

## Setup

Run the cell below once at the start of the session. It installs the `grover_workshop` support package used throughout this notebook.

In [ ]:
!pip install git+https://github.com/<your-org>/grover-workshop.git -q
# NOTE: replace <your-org> with the actual repository once it exists.

In [ ]:
# Built-in modules
import math
import random
import matplotlib.pyplot as plt

# Qiskit
from qiskit import QuantumCircuit
from qiskit.circuit.library import grover_operator, MCMTGate, ZGate
from qiskit.primitives import StatevectorSampler

# Workshop support package
from grover_workshop import (
    build_quantum_oracle,
    EXAMPLE_MARKED_STATES,
    quantum_query,
    angle_to_marked,
    get_secret_oracle_1,
    get_secret_oracle_2,
    grade_5_1_superposition,
    grade_5_2_single_layer,
    grade_5_3_readout,
    grade_5_4_iteration_count,
    grade_5_5_full_run,
    grade_6_1_query_intro,
    grade_6_2_secret_oracle,
    grade_6_3_bonus_discovery,
    grade_7_3_step_a,
    grade_7_3_step_a_exercise,
    grade_7_4_step_b,
    grade_7_5_step_c,
)

## 1. Intro to the oracle search problem

Imagine you're handed a black box. You can feed it any guess you like, and it answers only yes or no: *"is this the one?"* You can't look inside — you don't know how it decides, only that it decides correctly.

This is the **oracle search problem**: find whatever the box says yes to, using as few questions as possible.

Framing the problem this way — as a black box you can only *query*, not inspect — is deliberate. It lets us compare *how many questions* different approaches need, without needing to know anything about what's inside the box. That's the fair basis for comparison we'll use throughout this workshop.

The question this workshop answers: **classically, how many questions does this take — and can a quantum computer do better?** If so, how much better, and why?

**Roadmap:** we'll first look at the classical worst case, then see how a quantum computer can query the same kind of oracle, then meet the algorithm that turns those quantum queries into a real speedup — and finally, you'll build and run it yourself.

## 2. Classical oracle — worst case
*(shown by the instructor)*

First, let's see the classical version of the oracle search problem in code, and how many queries it takes in the worst case.

In [ ]:
def build_classical_oracle(marked_states):
    def oracle(guess):
        return guess in marked_states
    return oracle

marked_states = ["1101", "0001"]
classical_oracle = build_classical_oracle(marked_states)

print(classical_oracle("0000"))  # False
print(classical_oracle("1101"))  # True

How would we discover all the marked states underlying this oracle, if we didn't already know them? Classically, in the worst case, we need $\mathbf{O}(N)$ oracle queries, where $N = 2^n$ — we may have to check almost every possible input before we're done.

In [ ]:
def search_classical(n, oracle):
    found = []
    for i in range(2 ** n):
        guess = bin(i)[2:].zfill(n)
        if oracle(guess):
            found.append(guess)
    return found

print(search_classical(4, classical_oracle))

## 3. Quantum oracle — demo
*(shown, no hands-on)*

A quantum oracle marks states by flipping their **phase** rather than returning a classical yes/no. We build one below for the same marked states used above — the same oracle will be reused throughout Sections 3 and 5, so you'll see it again.

In [ ]:
oracle = build_quantum_oracle(EXAMPLE_MARKED_STATES, name="example_oracle", draw_circuit=True)

A phase flip alone isn't something you can directly measure. To actually observe whether a given state is marked, we interfere it against a reference state and measure which one "wins". This is what `quantum_query` does — you'll see the same mechanism again in Section 6.

In [ ]:
print("Query a marked state:", quantum_query(oracle, "1101", draw_circuit=True))
print("Query an unmarked state:", quantum_query(oracle, "0000"))

Notice the contrast: a marked guess measures almost entirely as `1`, an unmarked guess almost entirely as `0`. This is the signal Grover's algorithm amplifies, many queries' worth at once, using superposition.

## 4. Anatomy of Grover's algorithm

§1 asked: classically we need $O(N)$ queries — can a quantum computer do better? The answer has a name: **Grover's algorithm**.

### The rotation picture

Grover's algorithm has an exact geometric description. Define two vectors: $|\alpha\rangle$, the equal superposition of all *unmarked* states, and $|\beta\rangle$, the equal superposition of all *marked* states. Every state the algorithm ever produces — starting from the uniform superposition — stays exactly in the 2D plane spanned by $|\alpha\rangle$ and $|\beta\rangle$.

The starting state sits at
$$|\psi_0\rangle = \cos(\theta/2)\,|\alpha\rangle + \sin(\theta/2)\,|\beta\rangle, \qquad \sin(\theta/2) = \sqrt{M/N}$$

and each Grover iteration rotates the state by a fixed angle $\theta$ toward $|\beta\rangle$:
$$|\psi_k\rangle = \cos\!\left(\frac{(2k+1)\theta}{2}\right)|\alpha\rangle + \sin\!\left(\frac{(2k+1)\theta}{2}\right)|\beta\rangle$$

In Section 5 you'll watch this rotation happen directly, one layer at a time, using a provided helper that measures exactly how close you are to $|\beta\rangle$.

### What one iteration does

Each Grover iteration (or *Grover operator*) is two steps in sequence:

- **The oracle** — flips the phase of marked states. You already saw this in Section 3.
- **The diffuser** — an operation sometimes called "inversion about the mean": it reflects every amplitude about the *average* amplitude. Combined with the oracle's phase flip, this has the effect of boosting the marked states' amplitudes and suppressing everyone else's, a little more with each iteration.

We won't build the diffuser by hand in this session — we'll use Qiskit's built-in `grover_operator()`, which bundles the oracle and diffuser together into one reusable operator.

### Optimal number of iterations

Since each iteration rotates the state by the *same* fixed angle $\theta$, there's a sweet spot: too few iterations and you haven't rotated far enough toward $|\beta\rangle$; too many and you rotate *past* it and start moving away again. The iteration count that lands closest to $|\beta\rangle$ is
$$k_{\text{opt}} = \left\lfloor \frac{\pi}{4\arcsin\sqrt{M/N}} \right\rfloor$$
You'll compute this yourself in Section 5, and then verify what happens if you deliberately overshoot it.

### The speedup, stated carefully

Grover's algorithm finds a marked state using $O(\sqrt{N})$ oracle queries, compared to the classical $O(N)$ — a quadratic reduction in the number of *queries* needed. This is an asymptotic, query-counting result for unstructured search: it does **not** mean that today's quantum hardware searches real-world problems faster than a classical computer in wall-clock time, and it isn't conditional on special structure in the problem — it holds generally for the black-box oracle model we've been using all session.

The original result is due to Lov K. Grover, *"A fast quantum mechanical algorithm for database search"*, Proceedings of the 28th Annual ACM Symposium on Theory of Computing (STOC '96), pp. 212–219 (freely available as [arXiv:quant-ph/9605043](https://arxiv.org/abs/quant-ph/9605043)). For a more approachable walkthrough, see [Qiskit's Grover's algorithm textbook chapter](https://learning.quantum.ibm.com/).

In [ ]:
grover_op = grover_operator(oracle)
grover_op.draw(output="mpl", style="iqp")

In [ ]:
M = len(EXAMPLE_MARKED_STATES)
N = 2 ** grover_op.num_qubits
optimal_num_iterations = math.floor(
    math.pi / (4 * math.asin(math.sqrt(M / N)))
)
print("Optimal number of iterations:", optimal_num_iterations)

## 5. Build Grover layer-by-layer
*(hands-on, core)*

We'll now build up the full algorithm ourselves, one piece at a time, using the same `oracle` from Section 3. Each step below has a `TODO` cell for you to complete, followed by a `grade_5_*` call that checks your work and explains what to fix if something's off.

### 5.1 — Equal superposition

In [ ]:
# TODO: build a circuit on `grover_op.num_qubits` qubits and apply H to every qubit
# so that all basis states are equally likely.
qc_5_1 = QuantumCircuit(grover_op.num_qubits)
# your code here

qc_5_1.draw(output="mpl", style="iqp")

In [ ]:
print(grade_5_1_superposition(qc_5_1))

### 5.2 — Apply one Grover layer

In [ ]:
# TODO: starting from qc_5_1, compose exactly one layer of the Grover operator
# (grover_op.power(1)) onto a *copy* of the circuit.
qc_5_2 = qc_5_1.copy()
# your code here

print("Angle to marked subspace:", angle_to_marked(qc_5_2, EXAMPLE_MARKED_STATES))

In [ ]:
print(grade_5_2_single_layer(qc_5_2))

### 5.3 — Measure and interpret

In [ ]:
# TODO: measure all qubits of qc_5_2 and run it through the sampler to get a
# distribution. Then plot a histogram of the top outcomes.
qc_5_3 = qc_5_2.copy()
# your code here (measure_all, run sampler, get counts as `dist_5_3`)

top = sorted(dist_5_3.items(), key=lambda x: x[1], reverse=True)[:8]
keys, values = zip(*top)
plt.bar(keys, values)
plt.xticks(rotation=45, ha="right")
plt.title("Distribution after 1 Grover layer")
plt.show()

In [ ]:
# Based on the histogram above, which bitstring(s) do you think are marked?
my_guessed_states = []  # e.g. ["1101", "0001"]
print(grade_5_3_readout(my_guessed_states, dist_5_3))

### 5.4 — Optimal number of iterations

In [ ]:
# TODO: compute the optimal number of iterations for this oracle,
# using the formula from Section 4.
my_optimal_n = None  # your code here

In [ ]:
print(grade_5_4_iteration_count(my_optimal_n))

### 5.5 — Full run: optimal depth vs. over-rotation

In [ ]:
# TODO: build and run two circuits: one using `my_optimal_n` layers, and one
# using `my_optimal_n + 2` layers. Store their measurement distributions as
# `dist_optimal` and `dist_overrotated`.

# your code here

In [ ]:
print(grade_5_5_full_run(dist_optimal, dist_overrotated))

### Comparison: randomised classical search
*(shown by the instructor)*

How does this stack up against a classical approach? Below, `classical_search_randomised` queries the *classical* oracle in a random order, one guess at a time, and tracks how many marked states have been found after each query — a fair classical baseline to set against Grover's layer-by-layer progress.

In [ ]:
def classical_search_randomised(oracle, n, seed=None):
    rng = random.Random(seed)
    all_states = [format(i, f"0{n}b") for i in range(2 ** n)]
    rng.shuffle(all_states)
    cumulative_hits = []
    hits_so_far = 0
    for guess in all_states:
        if oracle(guess):
            hits_so_far += 1
        cumulative_hits.append(hits_so_far)
    return cumulative_hits

classical_hits = classical_search_randomised(classical_oracle, grover_op.num_qubits, seed=42)

### Summary table

Putting it all together: for each Grover layer, the marked-state probability, the angle to the marked subspace, and (for comparison) how many marked states a randomised classical search would have found using the same number of queries.

In [ ]:
sampler = StatevectorSampler(seed=42)
rows = []
for k in range(my_optimal_n + 3):
    qc = QuantumCircuit(grover_op.num_qubits)
    qc.h(range(grover_op.num_qubits))
    qc.compose(grover_op.power(k), inplace=True)
    angle = angle_to_marked(qc, EXAMPLE_MARKED_STATES)

    qc_meas = qc.copy()
    qc_meas.measure_all()
    result = sampler.run([qc_meas], shots=2000).result()
    dist = result[0].data.meas.get_counts()
    p_marked = sum(dist.get(s, 0) for s in EXAMPLE_MARKED_STATES) / 2000

    classical_index = min(k, len(classical_hits) - 1)
    rows.append((k, round(p_marked, 3), round(angle, 3), classical_hits[classical_index]))

print(f"{'Layer':>6} {'P(marked)':>10} {'Angle (rad)':>12} {'Classical hits':>15}")
for row in rows:
    print(f"{row[0]:>6} {row[1]:>10} {row[2]:>12} {row[3]:>15}")

## 6. Guess the secret oracles
*(hands-on, core, graded)*

Now you'll apply what Section 5 taught to two oracles whose marked states you don't get to see — only what you can learn by querying them.

### 6.1 — Secret oracle 1: reinforcing the quantum query

`get_secret_oracle_1()` returns an 8-qubit oracle and **one** marked state, revealed to you. The number of *other* marked states is not disclosed — that's the Section 6.3 bonus.

Use the same single-query mechanism from Section 3 (`quantum_query`) to test:
(a) the revealed marked state, and
(b) some other bitstring you choose that you don't expect to be marked.

In [ ]:
secret_oracle_1, revealed_state = get_secret_oracle_1()
print("Revealed marked state:", revealed_state)

# TODO: query the revealed state, and query some other bitstring of your choosing.
result_correct = None    # your code here
result_incorrect = None  # your code here

In [ ]:
print(grade_6_1_query_intro(result_correct, result_incorrect))

### 6.2 — Secret oracle 2: main exercise

`get_secret_oracle_2()` returns a 5-qubit oracle and the **declared** number of marked states $M$ — the states themselves are hidden. Reuse the full Section 5 pipeline: superposition, compute the optimal iteration count from this $M$, apply that many layers, measure, and read off the marked bitstrings.

Grading here gives correct/incorrect only — no further detail — so you're not able to fish for the answer by trial and error.

In [ ]:
secret_oracle_2, declared_M = get_secret_oracle_2()
print("Declared number of marked states:", declared_M)

# TODO: build the full Grover pipeline for secret_oracle_2, using declared_M
# to compute the optimal iteration count. Measure and inspect the histogram.

# your code here

In [ ]:
# Based on your histogram, what are the marked states?
my_secret_2_guess = []  # e.g. ["00111", "00001", "10001"]
print(grade_6_2_secret_oracle(my_secret_2_guess))

### 6.3 — Bonus: discover the rest of secret oracle 1
*(optional, time permitting)*

Revisit secret oracle 1. You now know one of its marked states but not $M$, nor the rest. Apply the full search pipeline from Sections 5/6.2 (without knowing $M$ in advance, you'll need to try a few iteration counts, or sweep over several) to find the remaining marked states.

In [ ]:
# TODO: search secret_oracle_1 for its remaining marked states.

# your code here

In [ ]:
my_secret_1_guess = []  # all marked states you believe you've found, including the revealed one
print(grade_6_3_bonus_discovery(my_secret_1_guess))

## 7. *(Optional, self-paced)* Build your own oracle from scratch

This section is written to be worked through on your own, without an instructor walking through it live — whether that's during the workshop if you finish early, or afterwards. It stands alone: you don't need to remember every detail from earlier sections, though it does reuse code and variables from **Section 5**, so make sure those cells have been run first.

**A quick recap, since this section may be read separately from the rest:** an oracle marks certain basis states by flipping their phase. Section 3 showed you a working oracle being *used*. This section shows you how one gets *built*, starting from a single marked state and working up to the general case.

**Goal:** by the end of this section, you'll have built, from scratch, a function that constructs the exact same kind of oracle used throughout Sections 3 and 5 — for *any* list of marked states, not just the fixed example.

**A note before you start:** this section expects some comfort with basic programming — in particular, looping over a list, and finding positions of characters within a string (e.g. "which indices of this bitstring are `'0'`?"). If those are unfamiliar, that's alright — take it slowly, and treat working through them as part of what this section teaches.

### 7.3 — Step A: one marked state, by hand

Let's mark a single fixed state, `"1101"`, on 4 qubits — the same bitstring you've seen throughout Sections 3 and 5. Recall the construction: for each position where the marked state has a `'0'`, sandwich that qubit between X gates around a multi-controlled-Z. This flips the sign of exactly the `"1101"` basis state and leaves every other basis state alone.

Work out by hand which qubit positions need the X-sandwich for `"1101"` (remember: Qiskit's bit ordering is reversed relative to how the string reads left-to-right), and type those indices directly below.

In [ ]:
marked_state_a = "1101"
qc_7_3 = QuantumCircuit(4)

# TODO: apply X to the qubit(s) corresponding to the '0' position(s) of marked_state_a
# your code here

# TODO: apply a multi-controlled-Z across all 4 qubits
# your code here

# TODO: undo the X gate(s) you applied above
# your code here

qc_7_3.draw(output="mpl", style="iqp")

In [ ]:
#@title Solution — Step A (click ▶ to reveal) { display-mode: "form" }
marked_state_a_solution = "1101"
qc_7_3_solution = QuantumCircuit(4)

rev = marked_state_a_solution[::-1]
zero_inds = [i for i in range(4) if rev[i] == "0"]

qc_7_3_solution.x(zero_inds)
qc_7_3_solution.compose(MCMTGate(ZGate(), 3, 1), inplace=True)
qc_7_3_solution.x(zero_inds)

qc_7_3_solution.draw(output="mpl", style="iqp")

In [ ]:
print(grade_7_3_step_a(qc_7_3, marked_state=marked_state_a))

### 7.3b — Exercise: a different case

Repeat the same manual process for a **different** marked state on **5 qubits**: `"01101"`. You'll need to work out the zero positions yourself — the indices from Step A won't transfer directly.

In [ ]:
marked_state_a_ex = "01101"
qc_7_3b = QuantumCircuit(5)

# TODO: repeat the Step A process for marked_state_a_ex (5 qubits this time)
# your code here

qc_7_3b.draw(output="mpl", style="iqp")

In [ ]:
#@title Solution — Step A exercise (click ▶ to reveal) { display-mode: "form" }
marked_state_a_ex_solution = "01101"
qc_7_3b_solution = QuantumCircuit(5)

rev = marked_state_a_ex_solution[::-1]
zero_inds = [i for i in range(5) if rev[i] == "0"]

qc_7_3b_solution.x(zero_inds)
qc_7_3b_solution.compose(MCMTGate(ZGate(), 4, 1), inplace=True)
qc_7_3b_solution.x(zero_inds)

qc_7_3b_solution.draw(output="mpl", style="iqp")

In [ ]:
print(grade_7_3_step_a_exercise(qc_7_3b, marked_state=marked_state_a_ex))

### 7.4 — Step B: generalise to any single marked state

Turn the manual process above into a function that works for *any* bitstring you pass it. This is where the string-processing note above matters: you'll need to find the zero positions **programmatically** (e.g. by looping over the string's characters) rather than reading them off by eye.

In [ ]:
def build_oracle_single(marked_state):
    num_qubits = len(marked_state)
    qc = QuantumCircuit(num_qubits)

    # TODO: find the zero positions of marked_state programmatically
    # (remember the bit-order reversal), then apply the same
    # X - multi-controlled-Z - X sandwich as in Step A.
    # your code here

    return qc.to_gate()

In [ ]:
#@title Solution — Step B (click ▶ to reveal) { display-mode: "form" }
def build_oracle_single_solution(marked_state):
    num_qubits = len(marked_state)
    qc = QuantumCircuit(num_qubits)

    rev_target = marked_state[::-1]
    zero_inds = [i for i in range(num_qubits) if rev_target[i] == "0"]

    if zero_inds:
        qc.x(zero_inds)
    if num_qubits == 1:
        qc.z(0)
    else:
        qc.compose(MCMTGate(ZGate(), num_qubits - 1, 1), inplace=True)
    if zero_inds:
        qc.x(zero_inds)

    return qc.to_gate()

In [ ]:
print(grade_7_4_step_b(build_oracle_single))

### 7.5 — Step C: multiple marked states

Finally, extend Step B to accept a **list** of marked states, looping over each one and applying the same construction for each. Once this works, it's the same function as `build_quantum_oracle` from the workshop package — you'll have built, from scratch, exactly what's been powering Sections 3 and 5 all along.

In [ ]:
def build_oracle_multi(marked_states):
    num_qubits = len(marked_states[0])
    qc = QuantumCircuit(num_qubits)

    # TODO: loop over marked_states, applying the Step B construction for each
    # your code here

    return qc.to_gate()

In [ ]:
#@title Solution — Step C (click ▶ to reveal) { display-mode: "form" }
def build_oracle_multi_solution(marked_states):
    num_qubits = len(marked_states[0])
    qc = QuantumCircuit(num_qubits)

    for target in marked_states:
        rev_target = target[::-1]
        zero_inds = [i for i in range(num_qubits) if rev_target[i] == "0"]

        if zero_inds:
            qc.x(zero_inds)
        if num_qubits == 1:
            qc.z(0)
        else:
            qc.compose(MCMTGate(ZGate(), num_qubits - 1, 1), inplace=True)
        if zero_inds:
            qc.x(zero_inds)

    return qc.to_gate()

In [ ]:
print(grade_7_5_step_c(build_oracle_multi))

### 7.6 — Plug your oracle into Grover
*(optional)*

Take the oracle your own `build_oracle_multi` produces for `EXAMPLE_MARKED_STATES`, and run it through the exact Section 5 pipeline — `grover_operator`, superposition, layers, measurement — reusing the `my_optimal_n` you already computed there. You should see the same sharpening histogram you saw in Section 5, now produced by an oracle you built yourself.

In [ ]:
my_oracle = build_oracle_multi(EXAMPLE_MARKED_STATES)
my_grover_op = grover_operator(my_oracle)

qc_7_6 = QuantumCircuit(my_grover_op.num_qubits)
qc_7_6.h(range(my_grover_op.num_qubits))
qc_7_6.compose(my_grover_op.power(my_optimal_n), inplace=True)
qc_7_6.measure_all()

sampler = StatevectorSampler()
result = sampler.run([qc_7_6], shots=2000).result()
dist_7_6 = result[0].data.meas.get_counts()

top = sorted(dist_7_6.items(), key=lambda x: x[1], reverse=True)[:8]
keys, values = zip(*top)
plt.bar(keys, values)
plt.xticks(rotation=45, ha="right")
plt.title("Grover's algorithm running on your own oracle")
plt.show()